# Corrida definitiva — Deliverable 1
**60 casos balanceados · un modelo por vez · resultados guardados en Google Drive**

Ejecuta las celdas **en orden, de arriba abajo**. La única que vas a editar es la 5, para elegir qué modelo medir.

Los resultados se escriben en tu Drive, así que sobreviven aunque Colab se desconecte o destruya la máquina.

## 1 · GPU y dependencias
Tarda unos dos minutos. Debe decir `cuda: True`.

In [ ]:
!nvidia-smi -L
!pip -q install -U transformers accelerate bitsandbytes
import torch; print('cuda:', torch.cuda.is_available())

## 2 · Subir el paquete
Al ejecutar aparece un botón **Elegir archivos**. Sube `paquete_colab.zip` desde:

`C:\Lucas\Claude\2026-2\GenAI\deliverable-1\`

In [ ]:
from google.colab import files
files.upload();

## 3 · Descomprimir y verificar
Debe terminar con `9/9 casos de control correctos` y la distribución de los 60 casos.

In [ ]:
!unzip -o -q paquete_colab.zip -d proyecto
%cd /content/proyecto/scripts
!python verificador.py | tail -2
import json
from collections import Counter
casos = [json.loads(l) for l in open('../datos/casos.jsonl', encoding='utf-8') if l.strip()]
c = Counter(x['respuesta']['decision'] for x in casos)
print(f'\n{len(casos)} casos ·', dict(c))
print('mejor constante trivial:', f'{100*max(c.values())/len(casos):.1f}%')

## 4 · Conectar Google Drive
**Este es el paso que evita perder el trabajo.** Te va a pedir autorización: acepta con tu cuenta de Google.

Desde acá, todo lo que se mida se escribe en `MyDrive/genai_resultados/` en vez del disco temporal de Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/genai_resultados
!rm -rf /content/proyecto/resultados
!ln -s /content/drive/MyDrive/genai_resultados /content/proyecto/resultados

print('\nresultados guardados en Drive. Lo que ya hay:')
!ls -la /content/proyecto/resultados/

## 5 · Elegir el modelo
**Ésta es la única celda que editas.** Descomenta UNO por vez, corre la celda 6, y cuando termine vuelve acá y cambia al siguiente.

Orden recomendado — el primero es el que defiende los 3 puntos de bonus:

| # | Modelo | Params | Familia | Tiempo aprox. |
|---|---|---|---|---|
| 1 | Phi-3.5-mini-instruct | 3,8 B | Microsoft | ~15 min |
| 2 | Llama-3.1-8B-Instruct | 8,0 B | Meta | ~20 min · puede pedir licencia |
| 3 | Qwen2.5-7B-Instruct | 7,6 B | Alibaba | ~25 min |


In [ ]:
MODELO = 'microsoft/Phi-3.5-mini-instruct'
# MODELO = 'meta-llama/Llama-3.1-8B-Instruct'
# MODELO = 'Qwen/Qwen2.5-7B-Instruct'

# alternativas sin tramite, por si alguna de arriba falla:
# MODELO = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
# MODELO = 'Qwen/Qwen2.5-3B-Instruct'

print('modelo elegido:', MODELO)

## 6 · Medir
Corre las tres condiciones sobre los 60 casos. Si algo se corta, vuelve a ejecutar esta misma celda: retoma donde iba, no repite nada.

In [ ]:
import subprocess, sys, os

os.chdir('/content/proyecto/scripts')
CONDICIONES = [('zero_shot', False), ('zero_shot', True), ('few_shot', False)]

for cond, limpia in CONDICIONES:
    etq = cond + '/' + ('limpia' if limpia else 'prosa')
    print('=' * 68); print(MODELO, '·', etq); print('=' * 68)
    cmd = [sys.executable, 'runner.py', '--modelo', MODELO, '--condicion', cond]
    if limpia:
        cmd.append('--limpia')
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0:
        print('\n'.join(r.stdout.strip().splitlines()[-16:]))
    else:
        print('FALLO:')
        print('\n'.join(r.stderr.strip().splitlines()[-6:]))
        break
    print()

print('listo. archivos en Drive:')
!ls /content/proyecto/resultados/

## 7 · Tabla comparativa
Córrela cuando hayas medido todos los modelos que alcances. Ésta es la tabla del póster.

In [ ]:
import json, glob, os
from collections import Counter

casos = [json.loads(l) for l in open('/content/proyecto/datos/casos.jsonl', encoding='utf-8') if l.strip()]
cc = Counter(x['respuesta']['decision'] for x in casos)
trivial = 100 * max(cc.values()) / len(casos)

print('{:24} {:16} {:>3} {:>9} {:>8} {:>8} {:>6} {:>7}'.format(
    'modelo', 'condicion', 'n', 'decision', 'regla', 'ambas', 'cond.', 's/caso'))
print('-' * 88)
for f in sorted(glob.glob('/content/proyecto/resultados/*.raw.jsonl')):
    r = [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]
    if not r:
        continue
    nom = os.path.basename(f).replace('.raw.jsonl', '').split('__')
    p = lambda k: 100 * sum(x[k] for x in r) / len(r)
    ncond = sum(1 for x in r if x['prediccion']['decision'] == 'condicional')
    seg = sum(x['segundos'] for x in r) / len(r)
    print('{:24} {:16} {:3} {:8.1f}% {:7.1f}% {:7.1f}% {:6} {:6.1f}s'.format(
        nom[0][:24], nom[1] + '/' + nom[2], len(r),
        p('acierto_decision'), p('acierto_regla'), p('acierto_conjunto'), ncond, seg))
print('-' * 88)
print('{:24} {:16} {:3} {:8.1f}%'.format(
    'siempre lo mismo', '(sin leer nada)', len(casos), trivial))

## 8 · Taxonomía de errores
Qué tipo de error comete cada modelo. Alimenta el ítem de diagnóstico de la falla.

In [ ]:
import json, glob, os
from collections import Counter

malla = json.load(open('/content/proyecto/datos/malla.json', encoding='utf-8'))
codigos = {a['codigo'] for a in malla['asignaturas']}
validas = {'R-SIN-IMPEDIMENTO', 'R-DEPENDE-APROBACION', 'R-EXCEPCION-PRERREQ',
           'R-TOPE-MAX', 'R-TOPE-MIN', 'R-CREDITOS-MINIMOS', 'R-YA-CURSADA'}
esp = {'R-ESPECIAL-' + r for r in malla['requisitos_especiales_def']}
conocidas = validas | codigos | esp
NO_BLOQUEA = ('R-SIN-IMPEDIMENTO', 'R-DEPENDE-APROBACION', 'R-EXCEPCION-PRERREQ', None)

for f in sorted(glob.glob('/content/proyecto/resultados/*.raw.jsonl')):
    r = [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]
    if not r:
        continue
    print('=' * 68)
    print(os.path.basename(f).replace('.raw.jsonl', ''))
    print('=' * 68)
    print('  predijo         :', dict(Counter(x['prediccion']['decision'] for x in r)))
    reglas = Counter(x['prediccion']['regla'] for x in r)
    print('  reglas distintas:', len(reglas), '| mas citada:', reglas.most_common(1))
    contradice = sum(1 for x in r
                     if x['prediccion']['decision'] == 'sí'
                     and x['prediccion']['regla'] not in NO_BLOQUEA)
    print('  dice si citando una regla que bloquea:', contradice)
    inv = Counter(x['prediccion']['regla'] for x in r
                  if x['prediccion']['regla'] not in conocidas)
    print('  identificadores inventados:', dict(inv) if inv else 'ninguno')
    print('  parseo:', dict(Counter(x['prediccion']['parseo'] for x in r)))
    sub = [x for x in r if 'prerreq_en_curso' in x['rasgos']]
    if sub:
        print('  casos con prerrequisito en curso: {}/{}'.format(
            sum(x['acierto_decision'] for x in sub), len(sub)))
    print()

## 9 · Descargar
Opcional: los resultados ya están en tu Drive. Esto te los baja también al computador.

In [ ]:
import shutil
shutil.make_archive('/content/resultados', 'zip', '/content/drive/MyDrive/genai_resultados')
from google.colab import files
files.download('/content/resultados.zip')